In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import pandas as pd
import numpy as np

In [15]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

### Fix problem

### Confirm data time series overlap with 12442.  If data time series is the same then delete, if not then concatenate with 12442 into the Additional PRPA station (ENV Native ID 547) and delete

### Confirm data time series overlap with 12568.  If data time series is the same then delete 12568, if not then concatenate with 12568 into the Additional PRPA station (ENV Native ID 547) and delete.

### Additional (see stations 12568 and 12442)

In [16]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
"Confirm data time series overlap with 12442.  If data time series is the same then delete, if not then concatenate with 12442 into the Additional PRPA station (ENV Native ID 547) and delete"]

df_concat = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]


df_concat =  df_concat[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names']].reset_index(drop=True)

df_concat["Network ID"] = pd.to_numeric(df_concat["Network ID"], errors="coerce").astype("Int64")
df_concat["Station ID"] = pd.to_numeric(df_concat["Station ID"], errors="coerce").astype("Int64")

df_concat['check_station_id'] = ['12442']
df_concat




,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id
0,Confirm data time series overlap with 12442. ...,12568,9,BC-Air,Prince Rupert Fairview -> 547,NA -> Prince Rupert Fairview,12442


In [17]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND m_new.station_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(
    old_station_id,
    new_station_id,
    engine
):
    params = (
        # n_old
        old_station_id,

        # n_new
        new_station_id,

        # n_overlap
        old_station_id,
        new_station_id,

        # n_overlap_same_datum
        old_station_id,
        new_station_id
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]


stats = df_concat.apply(
    lambda r: count_station_stats(
        int(r["Station ID"]),
        int(r["check_station_id"]),
        engine
    ),
    axis=1
)

df_concat[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

df_concat

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,Confirm data time series overlap with 12442. ...,12568,9,BC-Air,Prince Rupert Fairview -> 547,NA -> Prince Rupert Fairview,12442,5660,168910,1148,1148


In [18]:
query = """
SELECT station_id
FROM meta_station
WHERE native_id = %s
"""

def get_new_native_id_from_s(native_id,engine):
    # handle <NA> safely
    if pd.isna(native_id):
        return None

    df = pd.read_sql(
        query,
        engine,
        params=(str(native_id),)
    )

    if df.empty:
        return None

    return df.iloc[0]["station_id"]  # TEXT → string

new_native_id = 547

df_concat["new_station_id_from_s"] = df_concat.apply(
    lambda row: get_new_native_id_from_s(
        new_native_id,
        engine
    ),
    axis=1
)
df_concat

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id,n_old,n_new,n_overlap,n_overlap_same_datum,new_station_id_from_s
0,Confirm data time series overlap with 12442. ...,12568,9,BC-Air,Prince Rupert Fairview -> 547,NA -> Prince Rupert Fairview,12442,5660,168910,1148,1148,2009


Seems the new native_id 547, exited in the current PCDS but under network 12 (FLNRO-WMB). So not sure how concatnate into the Additional PRPA station (ENV Native ID 547) (PRPA seems like the new station want to included).

Need to insert the new network, then new station, then concatnate data into the new station, then delete both stations

In [21]:

wanted_issues = [
    'Additional (see stations 12568 and 12442']

df_add = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]

df_add = df_add[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)

pattern = r'->\s*([0-9\.\-]+)\s*W,\s*([0-9\.\-]+)\s*N,\s*Elev\.\s*([0-9\.\-]+)\s*m'

df_add[['lon', 'lat', 'elev']] = df_add['Unique Location (String)'].str.extract(pattern).astype(float)

df_add["Native ID"] = df_add["Native ID"].str.replace(r'.*->\s*', '', regex=True)
df_add = df_add.drop(columns=['Unique Location (String)'])
df_add["Network Name"] = df_add["Network Name"].str.replace(r'.*->\s*', '', regex=True)


df_add['Station_name'] = df_add['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_add = df_add.drop(columns=['Unique Names'])
df_add.iloc[0]

from sqlalchemy import text

SQL_GET_NETWORK_ID = text("""
SELECT network_id
FROM meta_network
WHERE LOWER(network_name) = LOWER(:network_name)
""")

def get_network_id(network_name, engine):
    """Return the network_id for a given network_name."""
    with engine.connect() as conn:
        result = conn.execute(SQL_GET_NETWORK_ID, {"network_name": network_name}).scalar()
    return result

# Make sure 'Network Name' column exists
df_add['Network ID'] = df_add['Network Name'].apply(lambda name: get_network_id(name, engine))
df_add


,ISSUE,Station ID,Network ID,Network Name,Native ID,lon,lat,elev,Station_name
0,Additional (see stations 12568 and 12442,NaN,146,PRPA,547,130.352188,54.292615,15.0,Prince Rupert Fairview


In [29]:
from sqlalchemy import func

stations_created = []

# for _, row in df.iloc[0:2].iterrows():
for _, row in df_add.iterrows():
    
    name = row['Station_name']
    nid  = row['Native ID']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['Network ID'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['elev'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Prince Rupert Fairview', 15035)]


In [ ]:
import sqlalchemy as sa
from tqdm import tqdm

old_station_id = 15033

# --- SQL statements ---
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

# --- Execute deletions ---
with engine.begin() as conn:

    # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
    res_flags = conn.execute(delete_obs_flags, {"station_id": old_station_id})

    # 2️⃣ obs_raw
    res_obs = conn.execute(delete_obs, {"station_id": old_station_id})

    # 3️⃣ obs_derived_values (depends on meta_history)
    res_obs_dv = conn.execute(delete_obs_derived, {"station_id": old_station_id})

    # 4️⃣ meta_history
    res_hist = conn.execute(delete_history, {"station_id": old_station_id})

    # 5️⃣ meta_station
    res_sta = conn.execute(delete_station, {"station_id": old_station_id})

    # 6️⃣ Print summary
    print(
        f"Station {old_station_id}: "
        f"flags={res_flags.rowcount}, "
        f"obs_raw={res_obs.rowcount}, "
        f"obs_derived={res_obs_dv.rowcount}, "
        f"meta_history={res_hist.rowcount}, "
        f"meta_station={res_sta.rowcount}"
    )

Station 15033: flags=0, obs_raw=0, obs_derived=0, meta_history=0, meta_station=1


In [35]:
SQL_NEW_HISTORY = text("""
SELECT history_id
FROM meta_history
WHERE station_id = :new_station_id
ORDER BY history_id DESC
LIMIT 1
""")

from sqlalchemy import text

def move_station_obs_fast(old_station_id, new_station_id, engine):
    with engine.begin() as conn:

        # 1️⃣ get latest history_id for new station
        new_hist_id = conn.execute(
            SQL_NEW_HISTORY,
            {"new_station_id": new_station_id}
        ).scalar()

        if new_hist_id is None:
            print(f"New station {new_station_id} has no history, skipping.")
            return 0

        # 2️⃣ move observations
        n_move = conn.execute(
            text("""
                WITH updated AS (
                    UPDATE obs_raw o
                    SET history_id = :new_hist_id
                    FROM meta_history h_old
                    WHERE o.history_id = h_old.history_id
                      AND h_old.station_id = :old_station_id
                      AND o.obs_time >= DATE '2022-10-01'
                      AND NOT EXISTS (
                          SELECT 1
                          FROM obs_raw o_new
                          JOIN meta_history h_new
                            ON o_new.history_id = h_new.history_id
                          WHERE h_new.station_id = :new_station_id
                            AND o_new.obs_time = o.obs_time
                            AND o_new.vars_id  = o.vars_id
                      )
                    RETURNING 1
                )
                SELECT COUNT(*) FROM updated;
            """),
            {
                "old_station_id": old_station_id,
                "new_station_id": new_station_id,
                "new_hist_id": new_hist_id,
            }
        ).scalar()

        print(
            f"Moved {n_move} rows "
            f"from station {old_station_id} → {new_station_id}"
        )

        return n_move

old_station_id1 = 12442
old_station_id2 = 12568
new_station_id = 15035

print(f"Processing {old_station_id1} → {new_station_id}")
n = move_station_obs_fast(old_station_id1, new_station_id, engine)



Processing 12442 → 15035
Moved 0 rows from station 12442 → 15035


In [36]:
print(f"Processing {old_station_id2} → {new_station_id}")
n2 = move_station_obs_fast(old_station_id2, new_station_id, engine)

Processing 12568 → 15035
Moved 0 rows from station 12568 → 15035


### Check the new station's record

In [41]:
### Caution here, seems only one stations records there

In [39]:
query = """
SELECT COUNT(*) AS n_obs
FROM meta_history m
JOIN obs_raw o ON m.history_id = o.history_id
WHERE m.station_id = %s
"""

df = pd.read_sql(query, engine, params=(15035,))
print(df.iloc[0]["n_obs"])

65950


### delete old station data

In [37]:
import sqlalchemy as sa
from tqdm import tqdm

old_station_id = 12442

# --- SQL statements ---
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

# --- Execute deletions ---
with engine.begin() as conn:

    # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
    res_flags = conn.execute(delete_obs_flags, {"station_id": old_station_id})

    # 2️⃣ obs_raw
    res_obs = conn.execute(delete_obs, {"station_id": old_station_id})

    # 3️⃣ obs_derived_values (depends on meta_history)
    res_obs_dv = conn.execute(delete_obs_derived, {"station_id": old_station_id})

    # 4️⃣ meta_history
    res_hist = conn.execute(delete_history, {"station_id": old_station_id})

    # 5️⃣ meta_station
    res_sta = conn.execute(delete_station, {"station_id": old_station_id})

    # 6️⃣ Print summary
    print(
        f"Station {old_station_id}: "
        f"flags={res_flags.rowcount}, "
        f"obs_raw={res_obs.rowcount}, "
        f"obs_derived={res_obs_dv.rowcount}, "
        f"meta_history={res_hist.rowcount}, "
        f"meta_station={res_sta.rowcount}"
    )

Station 12442: flags=0, obs_raw=102960, obs_derived=0, meta_history=1, meta_station=1


In [38]:
import sqlalchemy as sa
from tqdm import tqdm

old_station_id = 12568

# --- SQL statements ---
delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

# --- Execute deletions ---
with engine.begin() as conn:

    # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
    res_flags = conn.execute(delete_obs_flags, {"station_id": old_station_id})

    # 2️⃣ obs_raw
    res_obs = conn.execute(delete_obs, {"station_id": old_station_id})

    # 3️⃣ obs_derived_values (depends on meta_history)
    res_obs_dv = conn.execute(delete_obs_derived, {"station_id": old_station_id})

    # 4️⃣ meta_history
    res_hist = conn.execute(delete_history, {"station_id": old_station_id})

    # 5️⃣ meta_station
    res_sta = conn.execute(delete_station, {"station_id": old_station_id})

    # 6️⃣ Print summary
    print(
        f"Station {old_station_id}: "
        f"flags={res_flags.rowcount}, "
        f"obs_raw={res_obs.rowcount}, "
        f"obs_derived={res_obs_dv.rowcount}, "
        f"meta_history={res_hist.rowcount}, "
        f"meta_station={res_sta.rowcount}"
    )

Station 12568: flags=0, obs_raw=5660, obs_derived=0, meta_history=1, meta_station=1


### Confirm data time series overlap with 12454, if the same then delete, if gap fill the concatenate into 12454, and then delete.


In [47]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
"Confirm data time series overlap with 12454, if the same then delete, if gap fill the concatenate into 12454, and then delete."]

df_concat = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]


df_concat =  df_concat[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names']].reset_index(drop=True)

df_concat["Network ID"] = pd.to_numeric(df_concat["Network ID"], errors="coerce").astype("Int64")
df_concat["Station ID"] = pd.to_numeric(df_concat["Station ID"], errors="coerce").astype("Int64")

df_concat['check_station_id'] = ['12454']
df_concat



,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id
0,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,Crofton Elementary -> 564,NA -> Crofton Elementary,12454


In [ ]:
from sqlalchemy import text

SQL_NEW_HISTORY = text("""
SELECT h.history_id
FROM meta_history h
WHERE h.station_id = :new_station_id
ORDER BY h.history_id DESC
LIMIT 1
""")

SQL_MOVE = text("""
WITH updated AS (
    UPDATE obs_raw o
    SET history_id = :new_hist_id
    FROM meta_history h_old
    WHERE o.history_id = h_old.history_id
      AND h_old.station_id = :old_station_id
      AND NOT EXISTS (
          SELECT 1
          FROM obs_raw o_new
          JOIN meta_history h_new ON o_new.history_id = h_new.history_id
          WHERE h_new.station_id = :new_station_id
            AND o_new.obs_time = o.obs_time
            AND o_new.vars_id  = o.vars_id
      )
    RETURNING 1
)
SELECT COUNT(*) FROM updated;
""")

def move_station_obs_fast(
    old_station_id,
    new_station_id,
    engine
):
    with engine.begin() as conn:

        new_hist_id = conn.execute(
            SQL_NEW_HISTORY,
            {"new_station_id": int(new_station_id)}
        ).scalar()

        if new_hist_id is None:
            print(f"New station {new_station_id} has no history, skipping.")
            return 0

        n_move = conn.execute(
            SQL_MOVE,
            {
                "old_station_id": int(old_station_id),
                "new_station_id": int(new_station_id),
                "new_hist_id": int(new_hist_id),
            }
        ).scalar()

        print(
            f"Moved {n_move} rows "
            f"from station {old_station_id} → {new_station_id}"
        )

        return n_move

results = []

for _, row in df_concat.iterrows():
    n = move_station_obs_fast(
        row["Station ID"],
        row["check_station_id"],
        engine
    )
    results.append(n)

df_concat["n_moved"] = results

Moved 4490 rows from station 12541 → 12454


NameError: name 'df_work' is not defined

In [52]:
df_concat

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,check_station_id,n_old,n_new,n_overlap,n_overlap_same_datum,n_moved
0,"Confirm data time series overlap with 12454, i...",12541,9,BC-Air,Crofton Elementary -> 564,NA -> Crofton Elementary,12454,5638,108436,1148,1148,4490


### The old station can be deleted

In [53]:
from tqdm import tqdm
import sqlalchemy as sa

station_ids = (
    df_concat["Station ID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)
# station_ids[0]


# List of station_ids to delete
station_ids_to_delete = station_ids  # or a subset

delete_obs_flags = sa.text("""
DELETE FROM obs_raw_pcic_flags fr
USING obs_raw o, meta_history h
WHERE fr.obs_raw_id = o.obs_raw_id
  AND o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs = sa.text("""
DELETE FROM obs_raw o
USING meta_history h
WHERE o.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_obs_derived = sa.text("""
DELETE FROM obs_derived_values dv
USING meta_history h
WHERE dv.history_id = h.history_id
  AND h.station_id = :station_id
""")

delete_history = sa.text("""
DELETE FROM meta_history
WHERE station_id = :station_id
""")

delete_station = sa.text("""
DELETE FROM meta_station
WHERE station_id = :station_id
""")

with engine.begin() as conn:
    for sid in tqdm(station_ids_to_delete, desc="Deleting stations"):

        # 1️⃣ obs_raw_pcic_flags (depends on obs_raw)
        res_flags = conn.execute(
            delete_obs_flags,
            {"station_id": sid}
        )

        # 2️⃣ obs_raw
        res_obs = conn.execute(
            delete_obs,
            {"station_id": sid}
        )

        # 3️⃣ obs_derived_values (depends on meta_history)
        res_obs_dv = conn.execute(
            delete_obs_derived,
            {"station_id": sid}
        )

        # 4️⃣ meta_history
        res_hist = conn.execute(
            delete_history,
            {"station_id": sid}
        )

        # 5️⃣ meta_station
        res_sta = conn.execute(
            delete_station,
            {"station_id": sid}
        )

        tqdm.write(
            f"Station {sid}: "
            f"flags={res_flags.rowcount}, "
            f"obs_raw={res_obs.rowcount}, "
            f"obs_derived={res_obs_dv.rowcount}, "
            f"meta_history={res_hist.rowcount}, "
            f"meta_station={res_sta.rowcount}"
        )

Deleting stations: 100%|██████████| 1/1 [00:02<00:00,  2.10s/it]

Station 12541: flags=0, obs_raw=1148, obs_derived=0, meta_history=1, meta_station=1


In [ ]:
### How about the native_id change and name_change?

### NOT on ENV LIST

In [60]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
"NOT on ENV LIST",
"PARKS CANADA"]

df_list = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]


df_list =  df_list[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names']].reset_index(drop=True)

df_list["Network ID"] = pd.to_numeric(df_list["Network ID"], errors="coerce").astype("Int64")
df_list["Station ID"] = pd.to_numeric(df_list["Station ID"], errors="coerce").astype("Int64")

df_list




,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names
0,NOT on ENV LIST,2486,5,BCH-GSO-HMP,OMI,Ominica R. ab Osilinka R. -> Omineca River abo...
1,NOT on ENV LIST,2488,5,BCH-GSO-HMP,OSP,Ospika R. ab Alley Ck -> Ospika River above Al...
2,PARKS CANADA,2458,5,BCH-GSO-HMP,FDL,Fidelty Mtn. -> Fidelity Mountain


In [61]:

query = """
SELECT s.native_id, m.station_name
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
WHERE s.station_id = %s
  AND s.network_id = %s
"""

def get_native_id_and_name(station_id, network_id, engine):
    df = pd.read_sql(
        query,
        engine,
        params=(str(station_id), int(network_id))
    )

    if df.empty:
        return None, None

    return df.iloc[0]["native_id"], df.iloc[0]["station_name"]


df_list[["old_native_id", "old_station_name"]] = df_list.apply(
    lambda row: pd.Series(
        get_native_id_and_name(
            row["Station ID"],
            row["Network ID"],
            engine
        )
    ),
    axis=1
)

df_list

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,old_native_id,old_station_name
0,NOT on ENV LIST,2486,5,BCH-GSO-HMP,OMI,Ominica R. ab Osilinka R. -> Omineca River abo...,OMI,Ominica R. ab Osilinka R.
1,NOT on ENV LIST,2488,5,BCH-GSO-HMP,OSP,Ospika R. ab Alley Ck -> Ospika River above Al...,OSP,Ospika R. ab Alley Ck
2,PARKS CANADA,2458,5,BCH-GSO-HMP,FDL,Fidelty Mtn. -> Fidelity Mountain,FDL,Fidelty Mtn.


In [ ]:
The old info are all correct, what we want to do with it?

In [ ]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl
wanted_issues = [
"CLOSED - Possibly Trail Butler park pre 2010"]

df_list = df[
    df["ISSUE"].str.strip().isin(wanted_issues) 
]


df_list =  df_list[['ISSUE', 'Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names']].reset_index(drop=True)

df_list["Network ID"] = pd.to_numeric(df_list["Network ID"], errors="coerce").astype("Int64")
df_list["Station ID"] = pd.to_numeric(df_list["Station ID"], errors="coerce").astype("Int64")

df_list




,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names
0,CLOSED - Possibly Trail Butler park pre 2010,2589,9,BC-Air,M114902,Trail


In [ ]:

SELECT s.*, m.*
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
WHERE m.station_name = 'Trail Butler park'


Trail Butler park


In [63]:

query = """
SELECT s.native_id, m.station_name
FROM meta_history AS m
JOIN meta_station AS s ON m.station_id = s.station_id
WHERE s.station_id = %s
  AND s.network_id = %s
"""

def get_native_id_and_name(station_id, network_id, engine):
    df = pd.read_sql(
        query,
        engine,
        params=(str(station_id), int(network_id))
    )

    if df.empty:
        return None, None

    return df.iloc[0]["native_id"], df.iloc[0]["station_name"]


df_list[["old_native_id", "old_station_name"]] = df_list.apply(
    lambda row: pd.Series(
        get_native_id_and_name(
            row["Station ID"],
            row["Network ID"],
            engine
        )
    ),
    axis=1
)

df_list

,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,old_native_id,old_station_name
0,CLOSED - Possibly Trail Butler park pre 2010,2589,9,BC-Air,M114902,Trail,M114902,Trail


In [ ]:
Site CLOSED - Move all data collected post October 1, 2022 to new ENV-ASP site "Breanda Mines" Line 1718

